# 01 Build dataset

Builds every cleaned file the chapters use, from `data/raw/` (Inside Airbnb) and `data/external/` (Zillow, Freddie Mac, Florida DOR, Census). The logic lives in `src/build_dataset.py`; this notebook runs it step by step and checks the results. Column meanings are in `docs/data_dictionary.md`.

Run this only if you have the raw files. Everyone else loads the finished files from the shared Drive folder.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve() / 'src'))
import build_dataset as bd
print(bd.RAW, bd.OUT)

## 1. Listings panel
Stacks the four snapshots, converts types, and adds host tiers, operator merging, ZIP codes, distances and flags.

In [ ]:
listings = bd.build_listings()

In [ ]:
latest = listings[listings.snapshot_date == '2026-09-22']
print(latest.host_tier.value_counts(normalize=True).round(3))
print(latest.broward_tier.value_counts(normalize=True).round(3))
latest[latest.operator_accounts > 1].drop_duplicates('operator_key')[['operator_key', 'operator_accounts', 'operator_listings']].sort_values('operator_listings', ascending=False).head(10)

In [ ]:
listings.groupby('snapshot_date')['world_cup_title'].sum()

## 2. Reviews
Combines the review files from all four snapshots (keeping each review once), adds word count, a language guess and listing attributes for grouping.

In [ ]:
reviews = bd.build_reviews()

## 3. Calendar
Monthly unavailable nights per listing for each snapshot, plus daily rows for nights Jun 1 to Sep 30, 2026 as the Dec, Mar and Jun scrapes saw them (the World Cup window).

In [ ]:
bd.build_calendar()

## 4. Outside data
Zillow home values and rents by ZIP, the 30-year mortgage rate by month, and arm's-length Broward home sales.

In [ ]:
bd.build_external()